In [1]:
!pip install plotly

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.6/15.6 MB 19.3 MB/s eta 0:00:0000:0100:01


In [2]:
!pip install "anywidget>=0.9.13"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.7/213.7 kB 4.1 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 2.3 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 477.3/477.3 kB 7.9 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 914.9/914.9 kB 12.5 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 24.6 MB/s eta 0:00:00:00:01


In [3]:
!pip install -U kaleido

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 10.9 MB/s eta 0:00:0000:0100:01


In [4]:
!pip install anndata==0.8.0

  Using cached anndata-0.8.0-py3-none-any.whl (96 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 20.6 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: h5py
    Found existing installation: h5py 2.10.0
    Uninstalling h5py-2.10.0:
      Successfully uninstalled h5py-2.10.0
  Attempting uninstall: anndata
    Found existing installation: anndata 0.7.8
    Uninstalling anndata-0.7.8:
      Successfully uninstalled anndata-0.7.8
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
samap 1.0.2 requires h5py<=2.10, but you have h5py 3.8.0 which is incompatible.
sam-algorithm 1.0.0 requires h5py<=2.10.0, but you have h5py 3.8.0 which is incompatible.


In [1]:
def gexpnorm(adata, gene_dict,level, gl, fin_name):
    gi = [gene_dict[i] for i in gl]
    exp = adata[:,gi].X.A
    expr = pd.DataFrame(exp,columns = gl, index = adata.obs_names)
    expr["cell_type"] = adata.obs[level].values
    mean_expr = expr.groupby("cell_type").mean()
    frac_expr = expr.groupby("cell_type").apply(
    lambda x: (x > 0).mean())
    mean_expr_norm = (mean_expr - mean_expr.min()) / (
    mean_expr.max() - mean_expr.min())
    mean_expr.columns = fin_name
    frac_expr.columns = fin_name
    mean_expr_norm.columns = fin_name
    return(mean_expr_norm, frac_expr)

In [2]:
def orthogroup_mapper(orthogroups, label):
    mapping = {}
    for index in orthogroups.index:
        gene_list = orthogroups.loc[index, label]
        if type(gene_list) != float:
            genes = gene_list.split(',')
            for item in genes:
                mapping[item] = index
    return(mapping)

In [3]:
from samap.mapping import SAMAP
from samap.analysis import (get_mapping_scores, GenePairFinder, transfer_annotations,
                            sankey_plot, chord_plot, CellTypeTriangles, 
                            ParalogSubstitutions, FunctionalEnrichment,
                            convert_eggnog_to_homologs, GeneTriangles)
from samalg import SAM
import pandas as pd
from Bio import SeqIO
from samap.utils import (save_samap, load_samap)
import scanpy as sc
import matplotlib.colors
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
from scipy import sparse 
from scipy import cluster
import seaborn as sns
import random
import sklearn
from scipy.stats import poisson
from sklearn.neighbors import KernelDensity
import time
import dill
from scipy.optimize import minimize
import pickle
import itertools
import os
import anndata as ad
import math
import csv
import plotly.express as px
from tqdm import tqdm
from collections import Counter

In [4]:
fn = '../../Active_SAM_joined/SAM_CJ_joined_v2_cleaned_03122025.h5ad'
sam_cj = SAM()
sam_cj.load_data(fn)
gene_dict_cj = {}
for i in range(len(sam_cj.adata.var_names)):
    gene_dict_cj[sam_cj.adata.var_names[i]] = i
gene_dict_cj['NaN'] = 'NaN'

In [5]:
fn = '../../Active_SAM_joined/SAM_AC_ncbi_soupx_cleaned_03122025.h5ad'
sam_ac = SAM()
sam_ac.load_data(fn)
gene_dict_ac = {}
for i in range(len(sam_ac.adata.var_names)):
    gene_dict_ac[sam_ac.adata.var_names[i]] = i
gene_dict_ac['NaN'] = 'NaN'

In [6]:
fn = '../../Active_SAM_joined/SAM_XT_joined_Slc17a6_cleaned_03122205.h5ad'
sam_xt = SAM()
sam_xt.load_data(fn)
gene_dict_xt = {}
for i in range(len(sam_xt.adata.var_names)):
    gene_dict_xt[sam_xt.adata.var_names[i]] = i
gene_dict_xt['NaN'] = 'NaN'

In [7]:
fn = '../../Active_SAM_joined/SAM_DR_ncbi_joined_cleaned_07172026.h5ad'
sam_dr = SAM()
sam_dr.load_data(fn)
gene_dict_dr = {}
for i in range(len(sam_dr.adata.var_names)):
    gene_dict_dr[sam_dr.adata.var_names[i]] = i
gene_dict_dr['NaN'] = 'NaN'

In [84]:
orthogroups = pd.read_csv('../../Vert_emapper_allorgs_08012026.tsv',delimiter='\t', index_col = 'Unnamed: 0')
mg_mapping_ortho = orthogroup_mapper(orthogroups, 'MM')
mo_mapping_ortho = orthogroup_mapper(orthogroups, 'MO')
cj_mapping_ortho = orthogroup_mapper(orthogroups, 'CJ')
ac_mapping_ortho = orthogroup_mapper(orthogroups, 'AC')
xt_mapping_ortho = orthogroup_mapper(orthogroups, 'XT')
dr_mapping_ortho = orthogroup_mapper(orthogroups,'DR')

In [85]:
#marker list
tested_markers =  {
 'cj_ac_1':        ['Crh', 'Ndnf', 'Trpc6'],
 'cj_ac_3':        ['Chrnb3', 'Drd1', 'Igf1', 'Mtnr1a', 'Ndnf', 'Sgcg'],
 'ac_xt_1':        ['Cck', 'Crh', 'Ddc', 'Kiss1r', 'Th', 'Trpc4', 'Trpc6'],
 'cj_xt_dr_2':     ['Gabrd', 'Gnrh2', 'Nmu', 'Slc1a6', 'Trpc4', 'Trpc6'],
 'cj_ac_xt_dr_4':  ['Crh', 'Igf1', 'Ndnf', 'Ngf', 'Npy2r', 'Trpc6', 'Trpm2'],
 'ac_xt_3':        ['Igf1', 'Trpm2'],
 'cj_ac_2':        ['Cck', 'Npy2r', 'Sgcg'],
 'cj_ac_4':        ['Chrnb3', 'Jcad', 'Tnik'],
 'ac_xt_2':        ['Cck', 'Col1a2', 'Sgcg'],
 'ac_xt_4':        ['Brs3', 'Igf1', 'Rps6ka2', 'Slc1a6'],
 'ac_dr_1':        ['Ndnf'],
 'cj_ac_xt_1':     ['Crh', 'Ddc', 'Gnrh2', 'Ndnf', 'Prokr1', 'Th', 'Tpo', 'Trpm2'],
 'cj_xt_dr_1':     ['Cck', 'Ddc', 'Drd1', 'Opn5', 'Rps6ka2', 'Tph1'],
 'cj_ac_xt_dr_1':  ['Cck', 'Rps6ka2', 'Tpo'],
 'cj_ac_xt_dr_3':  ['Cck', 'Chrnb3', 'Ndnf', 'Npy2r', 'Rps6ka2', 'Sgcg'],
}

mg_list = []
for item in tested_markers:
    mg_list = mg_list + tested_markers[item]

In [86]:
#TF list
tested_markers = {'cj_ac_1':['Meis2', 'Nr2f2','Dach1'],
                 'cj_ac_3':['Tfap2a','Otx2','Npas3'],
                 'ac_xt_1':['Bsx','Esr1','Hmx2'],
                 'cj_xt_dr_2':['Prdm16', 'Zic1', 'Zic4'],
                 'cj_ac_xt_dr_4':['Gata3','Tal1'],
                 'ac_xt_3':['Casz1', 'Esrrg', 'Nfia'],
                 'cj_ac_2':['Prox1','Nr3c2','Rora'],
                 'cj_ac_4':['Tfap2a','Barhl2','Zeb2'],
                 'ac_xt_2':['Insm1','Nr2e1','Zfp516'],
                 'ac_xt_4':['Bhlhe22', 'Foxg1','Tcf4'],
                 'ac_dr_1':['Neurod2','Pitx2','Satb1'],
                 'cj_ac_xt_1':['Pitx2', 'Sim1', 'Uncx'],
                 'cj_xt_dr_1':['Prdm13','Sp5','St18'],
                 'cj_ac_xt_dr_1':['Ebf3','Irx1','Lef1'],
                 'cj_ac_xt_dr_3':['Lhx2','Lhx9','Tcf7l2']}
mg_list = []
for item in tested_markers:
    mg_list = mg_list + tested_markers[item]

In [87]:
def remove_latter_duplicates(lst):
    seen = set()
    result = []
    for item in lst:
        if item not in seen:
            seen.add(item)
            result.append(item)
    return result

In [88]:
mg_list_cl = remove_latter_duplicates(mg_list)

In [89]:
def check_repeats(lst):
    counts = Counter(lst)
    repeats = {val: cnt for val, cnt in counts.items() if cnt > 1}
    
    if repeats:
        print(f"Found {len(repeats)} repeated value(s):")
        for val, cnt in repeats.items():
            print(f"  '{val}' appears {cnt} times")
    else:
        print("No repeated values found.")

    return repeats

In [90]:
check_repeats(mg_list_cl)

No repeated values found.


{}

In [91]:
mg_list = mg_list_cl

In [93]:
#ortholog table
otohomo = pd.read_csv('../../OTO_star_nothreshold_missing_le2_expressionthresh_08022026.tsv',delimiter='\t',index_col = 'MM')

In [94]:
#manually added Gnrh2 from the ortholog table because only found in 3 organisms not found in mouse
otohomo.loc['Gnrh2',:] = [np.nan,np.nan,'LOC134299345','gnrh2','gnrh2',3]

In [95]:
CJ_names = {i:otohomo.loc[i,'CJ'] for i in mg_list}
AC_names = {i:otohomo.loc[i,'AC'] for i in mg_list}
XT_names = {i:otohomo.loc[i,'XT'] for i in mg_list}
DR_names = {i:otohomo.loc[i,'DR'] for i in mg_list}

In [96]:
cl_CJ_names = {k:v for k,v in CJ_names.items() if not pd.isna(v) and v in gene_dict_cj}
cl_AC_names = {k:v for k,v in AC_names.items() if not pd.isna(v) and v in gene_dict_ac}
cl_XT_names = {k:v for k,v in XT_names.items() if not pd.isna(v) and v in gene_dict_xt}
cl_DR_names = {k:v for k,v in DR_names.items() if not pd.isna(v) and v in gene_dict_dr}

In [98]:
index_order = tested_markers.keys()

In [99]:
len(tested_markers.keys())

15

In [101]:
nonmam_cj = sam_cj.adata[sam_cj.adata.obs['ss_subclass_nounlabeled_nmm_cl_v4_nn'].isin(index_order)]
nonmam_ac = sam_ac.adata[sam_ac.adata.obs['ss_subclass_nounlabeled_nmm_cl_v4_nn'].isin(index_order)]
nonmam_xt = sam_xt.adata[sam_xt.adata.obs['ss_subclass_nounlabeled_nmm_cl_v4_nn'].isin(index_order)]
nonmam_dr = sam_dr.adata[sam_dr.adata.obs['ss_subclass_nounlabeled_nmm_cl_v4_nn'].isin(index_order)]

In [102]:
mean_expr_norm_cj, frac_cj = gexpnorm(nonmam_cj,gene_dict_cj,'ss_subclass_nounlabeled_nmm_cl_v4_nn',
                             [cl_CJ_names[i] for i in mg_list if i in cl_CJ_names],
                            [i for i in mg_list if i in cl_CJ_names])
mean_expr_norm_ac, frac_ac = gexpnorm(nonmam_ac,gene_dict_ac,'ss_subclass_nounlabeled_nmm_cl_v4_nn',
                             [cl_AC_names[i] for i in mg_list if i in cl_AC_names],
                            [i for i in mg_list if i in cl_AC_names])
mean_expr_norm_xt, frac_xt = gexpnorm(nonmam_xt,gene_dict_xt,'ss_subclass_nounlabeled_nmm_cl_v4_nn',
                             [cl_XT_names[i] for i in mg_list if i in cl_XT_names],
                            [i for i in mg_list if i in cl_XT_names])
mean_expr_norm_dr, frac_dr = gexpnorm(nonmam_dr,gene_dict_dr,'ss_subclass_nounlabeled_nmm_cl_v4_nn',
                             [cl_DR_names[i] for i in mg_list if i in cl_DR_names],
                            [i for i in mg_list if i in cl_DR_names])

In [103]:
mean_expr_norm_cj

,Meis2,Nr2f2,Dach1,Tfap2a,Otx2,Npas3,Esr1,Hmx2,Prdm16,Zic1,...,Uncx,Prdm13,Sp5,St18,Ebf3,Irx1,Lef1,Lhx2,Lhx9,Tcf7l2
cell_type,,,,,,,,,,,,,,,,,,,,,
cj_ac_1,1.000000,1.000000,0.611773,0.000000,0.007851,0.380727,1.000000,0.000000,0.025276,0.057567,...,0.017366,0.000000,0.000000,0.072277,0.022062,0.104586,0.000000,0.008496,0.008149,0.000000
cj_ac_2,0.032976,0.012522,0.068933,0.000000,0.000000,0.246643,0.000000,0.000000,0.000000,1.000000,...,0.000000,0.000000,0.000000,0.000000,0.035239,0.000000,0.129844,0.706006,0.732827,1.000000
cj_ac_3,0.118634,0.017708,0.468490,1.000000,0.930010,1.000000,0.025824,0.000000,0.006414,0.009991,...,1.000000,0.000000,0.252202,0.019893,0.000000,0.000000,0.643972,0.000000,0.000000,0.006864
cj_ac_4,0.075621,0.000000,0.124616,0.805960,0.290440,0.144718,0.088560,0.000000,0.012407,0.019764,...,0.000000,0.000000,0.000000,0.002475,1.000000,0.082914,0.667077,0.604834,0.553810,0.140147
cj_ac_xt_1,0.000000,0.440592,0.334570,0.005575,0.254677,0.682048,0.957819,0.162687,0.012235,0.026963,...,0.689849,0.002509,0.000000,0.036717,0.219715,0.069107,0.002242,0.021479,0.008184,0.000500
cj_ac_xt_dr_1,0.297153,0.371566,0.266296,0.142389,0.110876,0.449025,0.170437,0.104150,0.012688,0.090979,...,0.027633,0.006193,0.036841,0.019647,0.908865,1.000000,0.336258,0.902634,0.994570,0.569507
cj_ac_xt_dr_3,0.036327,0.001094,0.000000,0.000000,0.034704,0.096486,0.024638,0.417677,0.054681,0.023254,...,0.000000,0.000000,0.000000,0.027782,0.022609,0.000000,0.018697,1.000000,1.000000,0.910246
cj_ac_xt_dr_4,0.068874,0.563564,0.308637,0.459927,1.000000,0.427607,0.169285,0.271163,0.017371,0.068616,...,0.196812,0.005843,0.055783,0.383245,0.205178,0.315232,0.070860,0.006692,0.027125,0.524730
cj_xt_dr_1,0.015673,0.134969,1.000000,0.000000,0.065216,0.000000,0.198615,1.000000,0.060551,0.000000,...,0.079509,1.000000,1.000000,1.000000,0.005723,0.000000,1.000000,0.000000,0.047614,0.567855


In [104]:
mean_expr_norm_cj = mean_expr_norm_cj.loc[[i for i in index_order if i in sam_cj.adata.obs['ss_subclass_nounlabeled_nmm_cl_v4_nn'].unique()],:]
mean_expr_norm_ac = mean_expr_norm_ac.loc[[i for i in index_order if i in sam_ac.adata.obs['ss_subclass_nounlabeled_nmm_cl_v4_nn'].unique()],:]
mean_expr_norm_xt = mean_expr_norm_xt.loc[[i for i in index_order if i in sam_xt.adata.obs['ss_subclass_nounlabeled_nmm_cl_v4_nn'].unique()],:]
mean_expr_norm_dr = mean_expr_norm_dr.loc[[i for i in index_order if i in sam_dr.adata.obs['ss_subclass_nounlabeled_nmm_cl_v4_nn'].unique()],:]

In [105]:
mean_expr_norm_ac.columns

Index(['Meis2', 'Nr2f2', 'Dach1', 'Tfap2a', 'Otx2', 'Npas3', 'Bsx', 'Esr1',
       'Hmx2', 'Prdm16', 'Zic1', 'Zic4', 'Gata3', 'Tal1', 'Casz1', 'Esrrg',
       'Nfia', 'Prox1', 'Nr3c2', 'Rora', 'Barhl2', 'Zeb2', 'Insm1', 'Nr2e1',
       'Zfp516', 'Bhlhe22', 'Foxg1', 'Tcf4', 'Neurod2', 'Pitx2', 'Satb1',
       'Sim1', 'Uncx', 'Prdm13', 'Sp5', 'St18', 'Ebf3', 'Irx1', 'Lef1', 'Lhx2',
       'Lhx9', 'Tcf7l2'],
      dtype='object')

In [106]:
df = pd.DataFrame(columns = ['celltype','gene','avg exp','frac'])

In [107]:
cjl = []
for ct in mean_expr_norm_cj.index:
    for g in mean_expr_norm_cj.columns:
        cjl.append(['cj_'+ct,g,mean_expr_norm_cj.loc[ct,g],frac_cj.loc[ct,g]])
        
acl = []
for ct in mean_expr_norm_ac.index:
    for g in mean_expr_norm_ac.columns:
        acl.append(['ac_'+ct,g,mean_expr_norm_ac.loc[ct,g],frac_ac.loc[ct,g]])
        
xtl =[]
for ct in mean_expr_norm_xt.index:
    for g in mean_expr_norm_xt.columns:
        xtl.append(['xt_'+ct,g,mean_expr_norm_xt.loc[ct,g],frac_xt.loc[ct,g]])
        
drl = []
for ct in mean_expr_norm_dr.index:
    for g in mean_expr_norm_dr.columns:
        drl.append(['dr_'+ct,g,mean_expr_norm_dr.loc[ct,g],frac_dr.loc[ct,g]])

In [108]:
cjdf = pd.DataFrame(data = cjl,columns = ['celltype','gene','avg exp','frac'])
acdf = pd.DataFrame(data = acl,columns = ['celltype','gene','avg exp','frac'])
xtdf = pd.DataFrame(data = xtl,columns = ['celltype','gene','avg exp','frac'])
drdf = pd.DataFrame(data = drl,columns = ['celltype','gene','avg exp','frac'])

In [109]:
df = pd.concat([cjdf,acdf,xtdf,drdf],axis = 0)

In [110]:
df = df.fillna(0)

In [111]:
df

,celltype,gene,avg exp,frac
0,cj_cj_ac_1,Meis2,1.000000,0.643863
1,cj_cj_ac_1,Nr2f2,1.000000,0.766600
2,cj_cj_ac_1,Dach1,0.611773,0.839034
3,cj_cj_ac_1,Tfap2a,0.000000,0.000000
4,cj_cj_ac_1,Otx2,0.007851,0.002012
...,...,...,...,...
235,dr_cj_ac_xt_dr_3,Irx1,0.006604,0.001618
236,dr_cj_ac_xt_dr_3,Lef1,0.012991,0.004854
237,dr_cj_ac_xt_dr_3,Lhx2,1.000000,0.036408
238,dr_cj_ac_xt_dr_3,Lhx9,0.943786,0.275081


In [112]:
fin_order =[]
for item in index_order:
    if 'cj' in item:
        fin_order.append('cj_' + item)
    if 'ac' in item:
        fin_order.append('ac_' + item)
    if 'xt' in item:
        fin_order.append('xt_' + item)
    if 'dr' in item:
        fin_order.append('dr_' + item)

In [113]:
fin_order

['cj_cj_ac_1',
 'ac_cj_ac_1',
 'cj_cj_ac_3',
 'ac_cj_ac_3',
 'ac_ac_xt_1',
 'xt_ac_xt_1',
 'cj_cj_xt_dr_2',
 'xt_cj_xt_dr_2',
 'dr_cj_xt_dr_2',
 'cj_cj_ac_xt_dr_4',
 'ac_cj_ac_xt_dr_4',
 'xt_cj_ac_xt_dr_4',
 'dr_cj_ac_xt_dr_4',
 'ac_ac_xt_3',
 'xt_ac_xt_3',
 'cj_cj_ac_2',
 'ac_cj_ac_2',
 'cj_cj_ac_4',
 'ac_cj_ac_4',
 'ac_ac_xt_2',
 'xt_ac_xt_2',
 'ac_ac_xt_4',
 'xt_ac_xt_4',
 'ac_ac_dr_1',
 'dr_ac_dr_1',
 'cj_cj_ac_xt_1',
 'ac_cj_ac_xt_1',
 'xt_cj_ac_xt_1',
 'cj_cj_xt_dr_1',
 'xt_cj_xt_dr_1',
 'dr_cj_xt_dr_1',
 'cj_cj_ac_xt_dr_1',
 'ac_cj_ac_xt_dr_1',
 'xt_cj_ac_xt_dr_1',
 'dr_cj_ac_xt_dr_1',
 'cj_cj_ac_xt_dr_3',
 'ac_cj_ac_xt_dr_3',
 'xt_cj_ac_xt_dr_3',
 'dr_cj_ac_xt_dr_3']

In [114]:
test = [0,.1,.25,.4]
fin_ct = []
fin_gene = []
avg_exp = []
frac = []
for th in test:
    for item in df['celltype'].unique():
        fin_ct.append(item)
        fin_gene.append('test_' + str(th))
        avg_exp.append(1)
        frac.append(th)

In [115]:
test_df = pd.DataFrame([fin_ct,fin_gene,avg_exp,frac], index = ['celltype','gene','avg exp','frac']).T

In [116]:
df = pd.concat([df,test_df])

In [117]:
df

,celltype,gene,avg exp,frac
0,cj_cj_ac_1,Meis2,1.0,0.643863
1,cj_cj_ac_1,Nr2f2,1.0,0.7666
2,cj_cj_ac_1,Dach1,0.611773,0.839034
3,cj_cj_ac_1,Tfap2a,0.0,0.0
4,cj_cj_ac_1,Otx2,0.007851,0.002012
...,...,...,...,...
151,dr_cj_ac_xt_dr_4,test_0.4,1,0.4
152,dr_ac_dr_1,test_0.4,1,0.4
153,dr_cj_xt_dr_1,test_0.4,1,0.4
154,dr_cj_ac_xt_dr_1,test_0.4,1,0.4


In [118]:
df['frac'] = df['frac'].astype(float)
df['avg exp'] = df['avg exp'].astype(float)

In [119]:
df['frac'] =  df['frac'].clip(upper=.4)

In [120]:
df[df['celltype'] == 'cj_cj_ac_xt_dr_4']

,celltype,gene,avg exp,frac
117,cj_cj_ac_xt_dr_4,Meis2,0.068874,0.064919
118,cj_cj_ac_xt_dr_4,Nr2f2,0.563564,0.400000
119,cj_cj_ac_xt_dr_4,Dach1,0.308637,0.400000
120,cj_cj_ac_xt_dr_4,Tfap2a,0.459927,0.064919
121,cj_cj_ac_xt_dr_4,Otx2,1.000000,0.162297
122,cj_cj_ac_xt_dr_4,Npas3,0.427607,0.400000
123,cj_cj_ac_xt_dr_4,Esr1,0.169285,0.032459
124,cj_cj_ac_xt_dr_4,Hmx2,0.271163,0.002497
125,cj_cj_ac_xt_dr_4,Prdm16,0.017371,0.012484
126,cj_cj_ac_xt_dr_4,Zic1,0.068616,0.083645


In [122]:
fig = px.scatter(df, x = 'celltype', y = 'gene', size = 'frac', color = 'avg exp', color_continuous_scale= 'Blues', range_color=[0,1],opacity = 1,render_mode='pdf')
fig.update_xaxes(categoryorder='array', categoryarray= fin_order,
                range=[-0.5, len(df["celltype"].unique()) - 0.5])
fig.update_yaxes(categoryorder='array', categoryarray= mg_list + ['test_0','test_0.1','test_0.25','test_0.4'])
fig.update_layout(
    autosize=False,
    width=1800,
    height=1500,
)

max_size = 15

fig.update_traces(
    marker=dict(
        sizemode="area",
        sizeref=df["frac"].max() / max_size**2
    )
)

fig.update_xaxes(
    showgrid=False,
    showline=True,
    linewidth=1,
    linecolor="black",
    mirror=True,
)

fig.update_yaxes(
    showgrid=False,
    showline=True,
    linewidth=1,
    linecolor="black",
    mirror=True,
)
fig.update_layout(
    plot_bgcolor="white",
    paper_bgcolor="white",
)
fig.write_image("../../Figures/Figures_08022026/nonmammal_dotplot_test_TFexp_08042026.pdf")